# Tutorial 4: Disease Sample Annotation

Annotate disease/treatment samples — handle state changes and novel subtypes.

In [ ]:
import scanpy as sc
from mbanno import ConsensusAnnotator
from pathlib import Path

adata = sc.read_h5ad("data/disease_sample.h5ad")
print(f"Cells: {adata.n_obs:,}")

In [ ]:
annotator = ConsensusAnnotator(
    reference="allen-consensus-wmb",
    taxonomy_level="subclass"
)

# Lower confidence for disease samples to catch novel states
results = annotator.annotate(
    adata,
    methods=["marker"],
    min_confidence=0.4,  # relaxed threshold
)

In [ ]:
conf = results["confidence"]
print("Confidence distribution in disease samples:")
print(conf["confidence_level"].value_counts())
print(f"\nMean confidence: {conf['confidence'].mean():.3f}")

In [ ]:
# Check for 'novel_or_state' labels
annotations = results["annotations"]
flag_col = "confidence_level"
if flag_col in annotations.columns:
    flagged = annotations[annotations[flag_col].isin(["novel_or_state", "ambiguous"])]
    print(f"Flagged cells (potential novel states): {len(flagged)}/{len(annotations)}")

In [ ]:
annotator.save_results(results, Path("results/tutorial_04"))
annotator.generate_report(results, Path("results/tutorial_04"))

### Important Notes for Disease Annotation

- Use healthy reference (Allen WMB) first, annotate to subclass level
- Within each subclass, analyze state changes separately
- Don't confuse activation states with new cell types
- Examine marker support for any novel or ambiguous labels